# Bird Detector — Keras/TensorFlow (MobileNetV3-Large)
Binary classification: Bird (1) vs No-Bird (0)
Balcony bird detection with ONNX export for Android deployment.

In [ ]:
!pip install -q albumentations tf2onnx onnx onnxruntime

In [ ]:
import os, glob, json, random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Large

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, f1_score,
    accuracy_score, precision_score, recall_score
)

import warnings
warnings.filterwarnings('ignore')

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
CONFIG = {
    # Kaggle input paths
    'custom_birds_dir':    '/kaggle/input/datasets/sreekanthnanduri/balcony-bird-dataset/balcony-bird-dataset/birds',
    'custom_bg_dir':       '/kaggle/input/datasets/sreekanthnanduri/balcony-bird-dataset/balcony-bird-dataset/background',
    'kaggle_birds_dir':    '/kaggle/input/datasets/gpiosenka/birdies/images',
    'coco_img_dir':        '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017',
    'coco_ann_file':       '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_train2017.json',

    # Kaggle output paths
    'birds_aug_dir':       '/kaggle/working/birds_aug',
    'bg_aug_dir':          '/kaggle/working/background_aug',
    'models_dir':          '/kaggle/working/models',

    # Preprocessing
    'img_size':            224,
    'aug_multiplier':      5,

    # Training
    'batch_size':          32,
    'phase1_epochs':       5,
    'phase2_epochs':       20,
    'phase1_lr':           1e-3,
    'phase2_lr':           3e-5,   # lowered from 1e-4 — prevents Phase 2 overfitting
    'weight_decay':        1e-2,   # increased from 1e-4 — stronger regularization
    'val_split':           0.15,
    'test_split':          0.15,
    'seed':                42,
    'threshold':           0.5,

    # ImageNet normalization
    'mean': [0.485, 0.456, 0.406],
    'std':  [0.229, 0.224, 0.225],
}

for d in [CONFIG['birds_aug_dir'], CONFIG['bg_aug_dir'], CONFIG['models_dir']]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded.")
print(f"  Image size: {CONFIG['img_size']}x{CONFIG['img_size']}")
print(f"  Augmentation multiplier: {CONFIG['aug_multiplier']}x")
print(f"  Phase 1: {CONFIG['phase1_epochs']} epochs @ lr={CONFIG['phase1_lr']}")
print(f"  Phase 2: {CONFIG['phase2_epochs']} epochs @ lr={CONFIG['phase2_lr']} (CosineDecay)")

# Quick path check
print("\nChecking input paths...")
for key in ['custom_birds_dir', 'custom_bg_dir', 'kaggle_birds_dir', 'coco_img_dir']:
    path = CONFIG[key]
    status = "OK" if os.path.exists(path) else "NOT FOUND"
    print(f"  [{status}] {key}: {path}")

---
## Section 1 — Augmentation
Resize all custom images to 224×224 (BILINEAR, no cropping) then generate 5 augmented versions per image saved to disk.

In [ ]:
def get_aug_pipeline():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        # Asymmetric: allow brightening up to +20%, limit darkening to -5%
        # Prevents dark source images from going all-black
        A.RandomBrightnessContrast(
            brightness_limit=(-0.05, 0.20),
            contrast_limit=(-0.05, 0.15),
            p=0.5
        ),
        A.GaussianBlur(blur_limit=(3, 5), sigma_limit=0, p=0.3),
        # border_mode=4 = cv2.BORDER_REFLECT_101 — no black fill on rotation edges
        A.Rotate(limit=15, border_mode=4, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        # GaussNoise removed — albumentations 2.x treats var_limit in float space
        # causing completely noisy images. Use manual noise below instead.
    ])

def load_resize_image(img_path, size=224):
    """Load PNG/JPG and BILINEAR resize to (size x size). No cropping."""
    img = Image.open(img_path).convert('RGB')
    img = img.resize((size, size), Image.BILINEAR)
    return np.array(img)

def augment_and_save(src_dir, dst_dir, multiplier=5, img_size=224):
    """
    For each image in src_dir:
      - Resize to img_size x img_size (BILINEAR)
      - Generate `multiplier` augmented versions
      - Save all to dst_dir as PNG
      - Skips output if augmented image is nearly black (mean < 15)
    """
    os.makedirs(dst_dir, exist_ok=True)

    # Clear old augmented images to avoid stale bad results
    old_files = glob.glob(os.path.join(dst_dir, '*.png'))
    for f in old_files:
        os.remove(f)
    if old_files:
        print(f"Cleared {len(old_files)} old augmented images from {dst_dir}/")

    patterns = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
    img_paths = []
    for pat in patterns:
        img_paths.extend(glob.glob(os.path.join(src_dir, pat)))
    img_paths = list(set(img_paths))

    print(f"Found {len(img_paths)} images in {os.path.basename(src_dir)}/")

    aug = get_aug_pipeline()
    saved, skipped = 0, 0

    for img_path in img_paths:
        base = os.path.splitext(os.path.basename(img_path))[0]
        img  = load_resize_image(img_path, img_size)

        for i in range(multiplier):
            augmented = aug(image=img)['image'].copy()

            # Subtle manual noise (std=3 out of 255 — barely visible)
            # Avoids albumentations version compatibility issues with GaussNoise
            if np.random.rand() < 0.2:
                noise = np.random.normal(0, 3, augmented.shape).astype(np.int16)
                augmented = np.clip(augmented.astype(np.int16) + noise, 0, 255).astype(np.uint8)

            # Quality check: skip nearly-black images
            if augmented.mean() < 15:
                skipped += 1
                continue

            save_path = os.path.join(dst_dir, f"{base}_aug{i+1}.png")
            Image.fromarray(augmented).save(save_path)
            saved += 1

    print(f"Saved {saved} augmented images -> {dst_dir}/")
    if skipped:
        print(f"  Skipped {skipped} images (nearly black — source image too dark)")
    return saved

In [ ]:
print("=" * 50)
print("Augmenting custom bird images...")
print("=" * 50)
augment_and_save(
    CONFIG['custom_birds_dir'],
    CONFIG['birds_aug_dir'],
    multiplier=CONFIG['aug_multiplier'],
    img_size=CONFIG['img_size']
)

print()
print("=" * 50)
print("Augmenting custom background images...")
print("=" * 50)
augment_and_save(
    CONFIG['custom_bg_dir'],
    CONFIG['bg_aug_dir'],
    multiplier=CONFIG['aug_multiplier'],
    img_size=CONFIG['img_size']
)

---
## Section 2 — Dataset Loading & Balancing

In [ ]:
# Verify augmented image quality — check for black/noisy images before training
def show_aug_samples(aug_dir, n=8, title="Augmented Samples"):
    paths = sorted(glob.glob(os.path.join(aug_dir, '*.png')))
    if not paths:
        print(f"No images found in {aug_dir}")
        return
    sample = random.sample(paths, min(n, len(paths)))

    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    axes = axes.flatten()

    for i, path in enumerate(sample):
        img = Image.open(path).convert('RGB')
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(path)[:18], fontsize=7)
        axes[i].axis('off')
    for j in range(len(sample), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

print("Sample augmented BIRD images (verify no black/noisy images):")
show_aug_samples(CONFIG['birds_aug_dir'], n=8, title="Augmented Birds — Sample Check")

print("Sample augmented BACKGROUND images:")
show_aug_samples(CONFIG['bg_aug_dir'], n=8, title="Augmented Background — Sample Check")

In [ ]:
def collect_images(directory, extensions=('*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.JPEG')):
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(directory, ext)))
    return list(set(paths))

def load_kaggle_birds(birds_dir):
    """
    Birdies dataset (gpiosenka): flat folder of JPGs — no subdirectories.
    Label all as bird=1.
    """
    if not os.path.exists(birds_dir):
        print(f"WARNING: Kaggle birds dir not found: {birds_dir}")
        return []
    all_paths = collect_images(birds_dir)
    print(f"Kaggle birdies dataset: {len(all_paths)} images loaded")
    return all_paths

def load_coco_images(coco_img_dir, coco_ann_file):
    """
    Parse COCO 2017 annotations ONCE and return both:
      - bird_paths    : images containing at least one bird (category_id=16) → bird class
      - no_bird_paths : images with no bird annotations → no-bird class
    Avoids double-parsing the large JSON file.
    """
    if not os.path.exists(coco_ann_file):
        print(f"WARNING: COCO annotation file not found: {coco_ann_file}")
        return [], []

    print("Loading COCO annotations (this may take a moment)...")
    with open(coco_ann_file) as f:
        coco = json.load(f)

    # Find bird category id (always 16 in COCO but detect dynamically)
    bird_cat_id = None
    for cat in coco['categories']:
        if cat['name'] == 'bird':
            bird_cat_id = cat['id']
            break
    print(f"COCO bird category_id: {bird_cat_id}")

    # Collect image ids that contain at least one bird annotation
    bird_img_ids = set()
    for ann in coco['annotations']:
        if ann['category_id'] == bird_cat_id:
            bird_img_ids.add(ann['image_id'])
    print(f"COCO images containing birds: {len(bird_img_ids)}")

    # Split into bird / no-bird paths (check disk existence once)
    bird_paths, no_bird_paths = [], []
    for img_info in coco['images']:
        full_path = os.path.join(coco_img_dir, img_info['file_name'])
        if not os.path.exists(full_path):
            continue
        if img_info['id'] in bird_img_ids:
            bird_paths.append(full_path)
        else:
            no_bird_paths.append(full_path)

    print(f"COCO bird images (on disk):    {len(bird_paths)}")
    print(f"COCO no-bird images (on disk): {len(no_bird_paths)}")
    return bird_paths, no_bird_paths

# ----- Load all paths -----
print("Loading bird image paths...")
custom_bird_paths     = collect_images(CONFIG['custom_birds_dir'])
custom_bird_aug_paths = collect_images(CONFIG['birds_aug_dir'])
kaggle_bird_paths     = load_kaggle_birds(CONFIG['kaggle_birds_dir'])

print(f"\nCustom birds (original):  {len(custom_bird_paths)}")
print(f"Custom birds (augmented): {len(custom_bird_aug_paths)}")
print(f"Kaggle birdies:           {len(kaggle_bird_paths)}")

print("\nLoading COCO images (bird + no-bird, single JSON parse)...")
coco_bird_paths, coco_no_bird_paths = load_coco_images(CONFIG['coco_img_dir'], CONFIG['coco_ann_file'])

print("\nLoading background image paths...")
custom_bg_paths     = collect_images(CONFIG['custom_bg_dir'])
custom_bg_aug_paths = collect_images(CONFIG['bg_aug_dir'])

print(f"\nCustom background (original):  {len(custom_bg_paths)}")
print(f"Custom background (augmented): {len(custom_bg_aug_paths)}")

In [ ]:
def balance_classes(bird_paths, no_bird_paths, seed=42):
    random.seed(seed)
    min_count        = min(len(bird_paths), len(no_bird_paths))
    bird_balanced    = random.sample(bird_paths,    min_count)
    no_bird_balanced = random.sample(no_bird_paths, min_count)
    print(f"Balanced dataset:")
    print(f"  Bird class:    {min_count} images")
    print(f"  No-bird class: {min_count} images")
    print(f"  Total:         {min_count * 2} images")
    return bird_balanced, no_bird_balanced

# Combine all sources
# Bird class: custom (original + augmented) + Kaggle birdies + COCO bird images
all_bird_paths    = (custom_bird_paths + custom_bird_aug_paths +
                     kaggle_bird_paths + coco_bird_paths)

# No-bird class: custom background (original + augmented) + COCO no-bird images
all_no_bird_paths = (custom_bg_paths + custom_bg_aug_paths +
                     coco_no_bird_paths)

print(f"Bird sources:")
print(f"  Custom birds (original):  {len(custom_bird_paths)}")
print(f"  Custom birds (augmented): {len(custom_bird_aug_paths)}")
print(f"  Kaggle birdies:           {len(kaggle_bird_paths)}")
print(f"  COCO bird images:         {len(coco_bird_paths)}")
print(f"  Total bird:               {len(all_bird_paths)}")
print(f"\nNo-bird sources:")
print(f"  Custom background (original):  {len(custom_bg_paths)}")
print(f"  Custom background (augmented): {len(custom_bg_aug_paths)}")
print(f"  COCO no-bird images:           {len(coco_no_bird_paths)}")
print(f"  Total no-bird:                 {len(all_no_bird_paths)}")

bird_balanced, no_bird_balanced = balance_classes(all_bird_paths, all_no_bird_paths, seed=CONFIG['seed'])

all_paths  = bird_balanced + no_bird_balanced
all_labels = [1] * len(bird_balanced) + [0] * len(no_bird_balanced)

combined = list(zip(all_paths, all_labels))
random.seed(CONFIG['seed'])
random.shuffle(combined)
all_paths, all_labels = zip(*combined)
all_paths, all_labels = list(all_paths), list(all_labels)

# Train/val/test split
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG['test_split'],
    stratify=all_labels,
    random_state=CONFIG['seed']
)

val_ratio = CONFIG['val_split'] / (1 - CONFIG['test_split'])
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels,
    test_size=val_ratio,
    stratify=train_val_labels,
    random_state=CONFIG['seed']
)

print(f"\nTrain set : {len(train_paths):>6}  (Bird: {sum(train_labels)}, No-Bird: {len(train_labels)-sum(train_labels)})")
print(f"Val set   : {len(val_paths):>6}  (Bird: {sum(val_labels)}, No-Bird: {len(val_labels)-sum(val_labels)})")
print(f"Test set  : {len(test_paths):>6}  (Bird: {sum(test_labels)}, No-Bird: {len(test_labels)-sum(test_labels)})")

---
## Section 3 — tf.data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label, img_size=224):
    """Load image, BILINEAR resize to img_size, ImageNet normalize."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_size, img_size], method='bilinear')
    img = tf.cast(img, tf.float32) / 255.0
    # ImageNet normalization
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
    std  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
    img  = (img - mean) / std
    label = tf.cast(label, tf.float32)
    return img, label

def make_dataset(paths, labels, img_size, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=42)
    ds = ds.map(lambda p, l: load_and_preprocess(p, l, img_size), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, CONFIG['img_size'], CONFIG['batch_size'], shuffle=True)
val_ds   = make_dataset(val_paths,   val_labels,   CONFIG['img_size'], CONFIG['batch_size'], shuffle=False)
test_ds  = make_dataset(test_paths,  test_labels,  CONFIG['img_size'], CONFIG['batch_size'], shuffle=False)

print(f"Train batches: {len(train_ds)}")
print(f"Val batches:   {len(val_ds)}")
print(f"Test batches:  {len(test_ds)}")

# Class weight for imbalance handling
n_pos = sum(train_labels)
n_neg = len(train_labels) - n_pos
class_weight = {0: 1.0, 1: n_neg / n_pos}
print(f"\nClass weights — 0 (no-bird): {class_weight[0]:.4f}, 1 (bird): {class_weight[1]:.4f}")

---
## Section 4 — MobileNetV3-Large Model Setup

In [ ]:
def create_model(img_size=224):
    """
    MobileNetV3-Large pretrained on ImageNet.
    Classifier head replaced: GlobalAvgPool -> Dropout(0.4) -> Dense(1, sigmoid)
    """
    base_model = MobileNetV3Large(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'          # Global average pooling built-in
    )
    base_model.trainable = False  # Frozen for Phase 1

    inputs  = keras.Input(shape=(img_size, img_size, 3))
    x       = base_model(inputs, training=False)
    x       = layers.Dropout(0.4)(x)   # increased from 0.2 — stronger regularization
    outputs = layers.Dense(1, activation='sigmoid', name='bird_output')(x)

    model = Model(inputs, outputs, name='bird_detector_mobilenetv3')
    return model, base_model

model, base_model = create_model(CONFIG['img_size'])

total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (Phase 1: head only)")
model.summary(line_length=80)

---
## Section 5 — Training
### Phase 1: Frozen backbone — train classifier head only
### Phase 2: Unfreeze all — full fine-tune

In [ ]:
def get_callbacks(checkpoint_path, phase_name, reduce_lr=True):
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
    ]
    if reduce_lr:
        callbacks.append(
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-7,
                verbose=1
            )
        )
    return callbacks

In [ ]:
# Phase 1: backbone frozen, train head only
base_model.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['phase1_lr']),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc'),
    ]
)

trainable_params_p1 = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Phase 1 — Trainable parameters: {trainable_params_p1:,} (head only)")
print("=" * 60)
print("PHASE 1 — Training classifier head (backbone frozen)")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG['phase1_epochs'],
    class_weight=class_weight,
    callbacks=get_callbacks(
        os.path.join(CONFIG['models_dir'], 'best_phase1.keras'),
        'Phase 1'
    ),
    verbose=1
)

In [ ]:
# Phase 2: unfreeze all layers — full fine-tune with CosineDecay + gradient clipping
base_model.trainable = True

# CosineDecay: smoothly reduces LR from phase2_lr → phase2_lr * 0.01 over all epochs
total_steps = CONFIG['phase2_epochs'] * len(train_ds)
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=CONFIG['phase2_lr'],   # 3e-5
    decay_steps=total_steps,
    alpha=0.01   # final LR = 0.01 * 3e-5 = 3e-7
)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=lr_schedule,
        weight_decay=CONFIG['weight_decay'],   # 1e-2 — strong L2 regularization
        clipnorm=1.0                           # gradient clipping — prevents explosion
    ),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc'),
    ]
)

trainable_params_p2 = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Phase 2 — Trainable parameters: {trainable_params_p2:,} (all layers)")
print("=" * 60)
print("PHASE 2 — Full fine-tune (all layers unfrozen)")
print("=" * 60)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG['phase2_epochs'],
    class_weight=class_weight,
    callbacks=get_callbacks(
        os.path.join(CONFIG['models_dir'], 'best_phase2.keras'),
        'Phase 2',
        reduce_lr=False   # CosineDecay handles LR — ReduceLROnPlateau is incompatible
    ),
    verbose=1
)

---
## Section 6 — Evaluation on Test Set

In [ ]:
def evaluate_model(model, test_ds, test_labels_list, threshold=0.5):
    all_probs = model.predict(test_ds, verbose=1).flatten()
    all_preds = (all_probs >= threshold).astype(int)
    all_labels_arr = np.array(test_labels_list)

    acc  = accuracy_score(all_labels_arr, all_preds)
    prec = precision_score(all_labels_arr, all_preds)
    rec  = recall_score(all_labels_arr, all_preds)
    f1   = f1_score(all_labels_arr, all_preds)
    fpr_vals, tpr_vals, _ = roc_curve(all_labels_arr, all_probs)
    roc_auc = auc(fpr_vals, tpr_vals)

    print("=" * 50)
    print("TEST SET EVALUATION")
    print("=" * 50)
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}  <- priority metric")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  AUC-ROC   : {roc_auc:.4f}")
    print()
    print("Classification Report:")
    print(classification_report(all_labels_arr, all_preds, target_names=['No Bird', 'Bird']))

    return all_labels_arr, all_probs, all_preds

test_labels_arr, test_probs, test_preds = evaluate_model(
    model, test_ds, test_labels, threshold=CONFIG['threshold']
)

---
## Section 7 — Evaluation Plots

In [ ]:
def plot_training_curves(h1, h2):
    ep1 = len(h1.history['loss'])
    ep2 = len(h2.history['loss'])

    train_loss = h1.history['loss']     + h2.history['loss']
    val_loss   = h1.history['val_loss'] + h2.history['val_loss']
    train_acc  = h1.history['accuracy']     + h2.history['accuracy']
    val_acc    = h1.history['val_accuracy'] + h2.history['val_accuracy']

    x = list(range(1, ep1 + ep2 + 1))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Training History — Keras MobileNetV3-Large', fontsize=14, fontweight='bold')

    axes[0].plot(x, train_loss, label='Train Loss',   color='steelblue',  linewidth=2)
    axes[0].plot(x, val_loss,   label='Val Loss',     color='tomato',     linewidth=2)
    axes[0].axvline(x=ep1, color='gray', linestyle='--', linewidth=1.5, label=f'Phase 2 Start (epoch {ep1+1})')
    axes[0].set_title('Loss', fontsize=12)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x, train_acc, label='Train Accuracy', color='seagreen',   linewidth=2)
    axes[1].plot(x, val_acc,   label='Val Accuracy',   color='darkorange', linewidth=2)
    axes[1].axvline(x=ep1, color='gray', linestyle='--', linewidth=1.5, label=f'Phase 2 Start (epoch {ep1+1})')
    axes[1].set_title('Accuracy', fontsize=12)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/keras_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: /kaggle/working/keras_training_curves.png")

plot_training_curves(history_phase1, history_phase2)

In [ ]:
def plot_confusion_matrix(labels, preds):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['No Bird', 'Bird'],
        yticklabels=['No Bird', 'Bird'],
        ax=ax, annot_kws={"size": 14}
    )
    ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    tn, fp, fn, tp = cm.ravel()
    fig.text(0.15, 0.02, f'TN={tn}  FP={fp}  FN={fn}  TP={tp}', ha='center', fontsize=11, color='gray')
    plt.tight_layout()
    plt.savefig('/kaggle/working/keras_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: /kaggle/working/keras_confusion_matrix.png")

plot_confusion_matrix(test_labels_arr, test_preds)

In [ ]:
def plot_roc(labels, probs):
    fpr, tpr, thresholds = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    optimal_idx    = np.argmax(tpr - fpr)
    optimal_thresh = thresholds[optimal_idx]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
    ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
    ax.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', zorder=5,
               label=f'Optimal Threshold = {optimal_thresh:.3f}')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
    ax.set_title('ROC Curve — Test Set', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/kaggle/working/keras_roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: /kaggle/working/keras_roc_curve.png")
    print(f"Optimal threshold: {optimal_thresh:.4f}")

plot_roc(test_labels_arr, test_probs)

In [ ]:
def plot_pr_curve(labels, probs):
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    pr_auc = auc(recall, precision)
    f1_scores  = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
    best_f1_idx = np.argmax(f1_scores)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall, precision, color='purple', lw=2.5, label=f'PR Curve (AUC = {pr_auc:.4f})')
    ax.scatter(recall[best_f1_idx], precision[best_f1_idx], color='red', zorder=5,
               label=f'Best F1 = {f1_scores[best_f1_idx]:.4f} @ thresh {thresholds[best_f1_idx]:.3f}')
    baseline = sum(labels) / len(labels)
    ax.axhline(y=baseline, color='gray', linestyle='--', lw=1.5, label=f'Baseline (P={baseline:.2f})')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curve — Test Set', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    plt.tight_layout()
    plt.savefig('/kaggle/working/keras_pr_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: /kaggle/working/keras_pr_curve.png")

plot_pr_curve(test_labels_arr, test_probs)

In [ ]:
def plot_sample_predictions(model, test_ds, test_labels_list, threshold=0.5, n=16):
    MEAN = np.array([0.485, 0.456, 0.406])
    STD  = np.array([0.229, 0.224, 0.225])

    imgs_collected, labels_collected, probs_collected = [], [], []
    label_idx = 0

    for imgs_batch, _ in test_ds:
        preds_batch = model.predict(imgs_batch, verbose=0).flatten()
        for i in range(len(imgs_batch)):
            imgs_collected.append(imgs_batch[i].numpy())
            probs_collected.append(float(preds_batch[i]))
            labels_collected.append(test_labels_list[label_idx])
            label_idx += 1
        if len(probs_collected) >= n:
            break

    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    fig.suptitle('Sample Predictions — Green=Correct  Red=Wrong', fontsize=13, fontweight='bold')
    axes = axes.flatten()

    for i in range(n):
        img = imgs_collected[i] * STD + MEAN
        img = np.clip(img, 0, 1)

        pred_label = 'Bird'    if probs_collected[i] >= threshold else 'No Bird'
        true_label = 'Bird'    if labels_collected[i] == 1        else 'No Bird'
        color      = 'green'   if pred_label == true_label         else 'red'

        axes[i].imshow(img)
        axes[i].set_title(
            f"True: {true_label}\nPred: {pred_label} ({probs_collected[i]:.2f})",
            color=color, fontsize=9, fontweight='bold'
        )
        axes[i].axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig('/kaggle/working/keras_sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: /kaggle/working/keras_sample_predictions.png")

plot_sample_predictions(model, test_ds, test_labels, threshold=CONFIG['threshold'], n=16)

---
## Section 8 — ONNX Export
Export trained MobileNetV3-Large to ONNX format for Android deployment using tf2onnx.
Input shape: `(None, 224, 224, 3)` — NHWC format (TensorFlow convention)

In [ ]:
import tf2onnx
import onnx
import onnxruntime as ort

def export_to_onnx(model, save_path, img_size=224):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Define input signature
    input_signature = (
        tf.TensorSpec((None, img_size, img_size, 3), tf.float32, name='input'),
    )

    print(f"Exporting ONNX model to: {save_path}")
    model_proto, _ = tf2onnx.convert.from_keras(
        model,
        input_signature=input_signature,
        output_path=save_path,
        opset=13
    )
    print("tf2onnx export complete.")

    # Verify ONNX model
    onnx_model = onnx.load(save_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model structure verified.")

    # Test with ONNX Runtime
    ort_session = ort.InferenceSession(save_path)
    dummy_input = np.random.randn(1, img_size, img_size, 3).astype(np.float32)
    ort_input   = {ort_session.get_inputs()[0].name: dummy_input}
    ort_output  = ort_session.run(None, ort_input)

    prob = float(ort_output[0][0][0])
    print(f"ONNX Runtime test inference: prob = {prob:.4f}")

    size_mb = os.path.getsize(save_path) / (1024 * 1024)
    print(f"ONNX model size: {size_mb:.2f} MB")
    print(f"\nExport complete: {save_path}")

onnx_path = os.path.join(CONFIG['models_dir'], 'bird_detector_keras.onnx')
export_to_onnx(model, onnx_path, img_size=CONFIG['img_size'])

In [ ]:
# Save in .h5 format (HDF5 — classic Keras format, widely compatible)
h5_path = os.path.join(CONFIG['models_dir'], 'bird_detector_keras.h5')
model.save(h5_path)
print(f"HDF5 model (.h5) saved    → {h5_path}  ({os.path.getsize(h5_path)/1024/1024:.2f} MB)")
print("  Load: model = tf.keras.models.load_model('bird_detector_keras.h5')")

# Save in .keras format (new native Keras format — recommended for TF2)
keras_path = os.path.join(CONFIG['models_dir'], 'bird_detector_keras.keras')
model.save(keras_path)
print(f"Keras model (.keras) saved → {keras_path}  ({os.path.getsize(keras_path)/1024/1024:.2f} MB)")
print("  Load: model = tf.keras.models.load_model('bird_detector_keras.keras')")

print("\nAll saved model formats:")
print("  .onnx  → Android deployment via ONNX Runtime")
print("  .h5    → HDF5, classic Keras format, works everywhere")
print("  .keras → native Keras format, recommended for TF2 fine-tuning")

---
## Summary

| Metric | Value |
|---|---|
| Model | MobileNetV3-Large (ImageNet pretrained) |
| Framework | Keras / TensorFlow |
| Input | 224x224x3 NHWC (BILINEAR resize, no crop) |
| Output | Sigmoid probability (threshold=0.5) |
| ONNX Path | `/kaggle/working/models/bird_detector_keras.onnx` |

### Output Files
- `/kaggle/working/models/bird_detector_keras.onnx` — Android deployment
- `/kaggle/working/keras_training_curves.png`
- `/kaggle/working/keras_confusion_matrix.png`
- `/kaggle/working/keras_roc_curve.png`
- `/kaggle/working/keras_pr_curve.png`
- `/kaggle/working/keras_sample_predictions.png`

### Note on ONNX Input Format
- PyTorch ONNX model expects **NCHW**: `(batch, 3, 224, 224)`
- Keras ONNX model expects **NHWC**: `(batch, 224, 224, 3)`
Keep this in mind when integrating into the Android app.